In [14]:
!pip install sagemaker U
import sagemaker
import boto3
sess= sagemaker.Session()

In [15]:
sagemaker_session_bucket=None
if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client('iam')
    role = iam.get_role(RoleName='sagemaker_execution_role')['Role']['Arn']

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker role arn: arn:aws:iam::700538667523:role/service-role/AmazonSageMaker-ExecutionRole-20251018T003244
sagemaker session region: us-east-1


In [ ]:
from sagemaker.huggingface import HuggingFaceModel

hub = {
    "HF_MODEL_ID": "ij98/resume-job-encoder2",
    "HF_TASK": "feature-extraction"
}

model = HuggingFaceModel(
    transformers_version="4.37.0",  # ✅ Compatible version
    pytorch_version="2.1.0",
    py_version="py310",
    env=hub,
    role=role
)

# ✅ Deploy using an instance instead of serverless
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium"   # ✅ You may change to t3.large if memory needed
)


-----------

In [37]:
predictor.endpoint_name


'huggingface-pytorch-inference-2025-10-18-17-54-17-978'

In [38]:
import boto3

sm_client = boto3.client("sagemaker")

endpoint_name = predictor.endpoint_name  # or replace with string name
response = sm_client.describe_endpoint(EndpointName=endpoint_name)

print("Endpoint Name:", endpoint_name)
print("Status:", response["EndpointStatus"])


Endpoint Name: huggingface-pytorch-inference-2025-10-18-17-54-17-978
Status: InService


In [28]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Example inputs
resumes = [
    "Experienced Python developer with 3 years in data analysis, pandas and scikit-learn, built ML pipelines.",
    "NLP engineer experienced in transformers, Hugging Face, and fine-tuning language models.",
    "Software engineer skilled in Java, Spring, and backend services; limited ML experience."
]
jobs = [
    "We need a Data Scientist skilled in Python, pandas, scikit-learn to build ML pipelines and analyze data.",
    "Seeking an NLP Engineer with experience fine-tuning transformers and using Hugging Face.",
    "Backend developer role requiring Java and Spring framework experience."
]

# ✅ FIXED: Get embeddings for each resume
resume_embeddings = []
for text in resumes:
    result = predictor.predict({"inputs": text})
    
    # Handle different response formats
    if isinstance(result, list):
        if isinstance(result[0], list):
            # Format: [[embedding]]
            embedding = result[0]
        else:
            # Format: [embedding]
            embedding = result
    elif isinstance(result, dict):
        # Some models return {'embeddings': [...]}
        embedding = result.get('embeddings', result.get('predictions', result))
    else:
        embedding = result
    
    resume_embeddings.append(np.array(embedding).flatten())

# ✅ FIXED: Get embeddings for each job
job_embeddings = []
for text in jobs:
    result = predictor.predict({"inputs": text})
    
    # Handle different response formats
    if isinstance(result, list):
        if isinstance(result[0], list):
            embedding = result[0]
        else:
            embedding = result
    elif isinstance(result, dict):
        embedding = result.get('embeddings', result.get('predictions', result))
    else:
        embedding = result
    
    job_embeddings.append(np.array(embedding).flatten())

# Debug: Check dimensions
print("✅ Resume embeddings:")
for i, emb in enumerate(resume_embeddings):
    print(f"Resume {i}: {len(emb)} dimensions")

print("\n✅ Job embeddings:")
for i, emb in enumerate(job_embeddings):
    print(f"Job {i}: {len(emb)} dimensions")

# ⚠️ CHECK: Ensure all embeddings have the same dimension
all_dims = [len(emb) for emb in resume_embeddings + job_embeddings]
if len(set(all_dims)) > 1:
    print(f"\n⚠️ WARNING: Embeddings have different dimensions: {set(all_dims)}")
    print("This suggests the model is returning inconsistent outputs.")
    print("Consider using a fixed embedding model or padding/truncating.")
    
    # OPTIONAL: Pad to maximum dimension (not recommended for production)
    max_dim = max(all_dims)
    resume_embeddings = [np.pad(emb, (0, max_dim - len(emb))) for emb in resume_embeddings]
    job_embeddings = [np.pad(emb, (0, max_dim - len(emb))) for emb in job_embeddings]
    print(f"\n✅ Padded all embeddings to {max_dim} dimensions")

# Convert to numpy arrays
resume_embeddings = np.array(resume_embeddings)
job_embeddings = np.array(job_embeddings)

# ✅ Calculate cosine similarity matrix
# Rows = resumes, Columns = jobs
similarity_matrix = cosine_similarity(resume_embeddings, job_embeddings)

print("\n" + "="*60)
print("📊 COSINE SIMILARITY MATRIX (Resume × Job)")
print("="*60)
print(f"{'':30} | Job 0 | Job 1 | Job 2")
print("-" * 60)
for i, resume in enumerate(resumes[:20]):  # Truncate display
    resume_short = resume[:27] + "..." if len(resume) > 30 else resume
    similarities = " | ".join([f"{similarity_matrix[i][j]:.3f}" for j in range(len(jobs))])
    print(f"Resume {i}: {resume_short:20} | {similarities}")

print("\n" + "="*60)
print("🎯 BEST MATCHES")
print("="*60)
for i, resume in enumerate(resumes):
    best_job_idx = np.argmax(similarity_matrix[i])
    best_score = similarity_matrix[i][best_job_idx]
    print(f"\nResume {i}: {resume[:60]}...")
    print(f"  → Best match: Job {best_job_idx} (similarity: {best_score:.3f})")
    print(f"  → Job: {jobs[best_job_idx][:60]}...")

print("\n" + "="*60)
print("📈 RANKED MATCHES FOR EACH RESUME")
print("="*60)
for i, resume in enumerate(resumes):
    print(f"\nResume {i}: {resume[:60]}...")
    # Get sorted job indices by similarity (descending)
    ranked_jobs = np.argsort(similarity_matrix[i])[::-1]
    for rank, job_idx in enumerate(ranked_jobs, 1):
        score = similarity_matrix[i][job_idx]
        print(f"  {rank}. Job {job_idx} (score: {score:.3f}): {jobs[job_idx][:50]}...")

✅ Resume embeddings:
Resume 0: 9600 dimensions
Resume 1: 7296 dimensions
Resume 2: 7296 dimensions

✅ Job embeddings:
Job 0: 10368 dimensions
Job 1: 6912 dimensions
Job 2: 4992 dimensions

⚠️ WARNING: Embeddings have different dimensions: {9600, 6912, 10368, 7296, 4992}
This suggests the model is returning inconsistent outputs.
Consider using a fixed embedding model or padding/truncating.

✅ Padded all embeddings to 10368 dimensions

📊 COSINE SIMILARITY MATRIX (Resume × Job)
                               | Job 0 | Job 1 | Job 2
------------------------------------------------------------
Resume 0: Experienced Python develope... | 0.318 | 0.103 | 0.152
Resume 1: NLP engineer experienced in... | 0.119 | 0.300 | 0.155
Resume 2: Software engineer skilled i... | 0.198 | 0.143 | 0.210

🎯 BEST MATCHES

Resume 0: Experienced Python developer with 3 years in data analysis, ...
  → Best match: Job 0 (similarity: 0.318)
  → Job: We need a Data Scientist skilled in Python, pandas, scikit-l...

Re

In [42]:
# ========================================
# UNCOMMENT TO TEST (after uploading PDFs)
# ========================================
score = match_resume_job_pdf(
    resume_pdf_path="user-default-efs/CV_Example (1).pdf",
    job_pdf_path="user-default-efs/NLP Engineer_Post.pdf"
)


📄 Reading resume: user-default-efs/CV_Example (1).pdf
💼 Reading job description: user-default-efs/NLP Engineer_Post.pdf

✅ Resume extracted: 4759 characters
✅ Job extracted: 2555 characters

⚠️  Resume too long (4759 chars)
   Truncating to 2000 chars to fit BERT's 512 token limit

⚠️  Job description too long (2555 chars)
   Truncating to 2000 chars to fit BERT's 512 token limit

🔄 Getting embeddings from SageMaker endpoint...

📊 Resume embedding: 115200 dimensions
📊 Job embedding: 139008 dimensions

⚠️  Dimension mismatch detected!
🔧 Padding to 139008 dimensions...
✅ Padded embeddings to 139008 dimensions

🎯 MATCHING RESULT
Similarity Score: 0.1048 (10.5%)

ℹ️  Note: Text was truncated for processing
   Resume: 4759 → 2000 chars
   Job: 2555 → 2000 chars

❌ POOR MATCH - Not recommended
   Recommendation: Reject application

📝 Resume Text Used (first 300 chars):
------------------------------------------------------------
Isabella
 
Kim
 
isabella@kim.com
 
•
 
(557)
 
300-9285
 
•
 